# 14 — Standard Library Power Tour

Goal: learn the “batteries included” modules that replace lots of third‑party dependencies.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: `itertools` (iteration building blocks)

Think: lazy sequences and combinators.

In [ ]:

import itertools as it

print(list(it.islice(it.count(10, 2), 5)))   # 10,12,14,16,18
print(list(it.chain([1,2], [3,4], "ab")))

xs = [1,2,3]
print(list(it.permutations(xs, 2)))


## 2.
L2: `functools` (function utilities)

Key tools:
- `lru_cache` (memoization)
- `partial`
- `reduce` (rare; but useful)

In [ ]:

from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n: int) -> int:
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

print([fib(i) for i in range(10)])


## 3.
L3: `datetime` (dates and times)

Use timezone-aware datetimes in real systems.
Stdlib supports `zoneinfo` (3.9+) for IANA timezones.

In [ ]:

from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

now_utc = datetime.now(timezone.utc)
now_ny = now_utc.astimezone(ZoneInfo("America/New_York"))
print("utc:", now_utc.isoformat())
print("ny :", now_ny.isoformat())
print("tomorrow:", (now_ny + timedelta(days=1)).date())


## 4.
L4: `pathlib`, `shutil`, and `glob`

Filesystem operations.

In [ ]:

from pathlib import Path
import shutil
import tempfile

with tempfile.TemporaryDirectory() as d:
    dp = Path(d)
    (dp / "a.txt").write_text("a", encoding="utf-8")
    (dp / "b.txt").write_text("b", encoding="utf-8")

    print("glob:", [p.name for p in dp.glob("*.txt")])

    shutil.copy(dp / "a.txt", dp / "a_copy.txt")
    print("after copy:", [p.name for p in dp.iterdir()])


## 5.
L5: `subprocess` (calling other programs)

Use `subprocess.run(..., check=True)` and avoid `shell=True` unless necessary.

In [ ]:

import subprocess, sys

result = subprocess.run([sys.executable, "-c", "print('hi from child')"], capture_output=True, text=True, check=True)
print(result.stdout.strip())


## 6.
L6: `argparse` (CLIs)

For complex CLIs, consider `click` or `typer` (third-party), but `argparse` is everywhere.

In [ ]:

import argparse

parser = argparse.ArgumentParser(prog="demo", add_help=False)
parser.add_argument("--count", type=int, default=2)
args = parser.parse_args([])  # empty argv for notebook demo
print(args.count)


## 7.
L7: `secrets` vs `random`

- `random` is for simulations, not security.
- `secrets` is for tokens/passwords/CSRF, etc.

In [ ]:

import random, secrets

random.seed(0)
print("random:", [random.randint(1, 10) for _ in range(5)])
print("token:", secrets.token_urlsafe(16))


## 8.
L8: Exercises

1. Use `itertools.groupby` to group consecutive equal items.
2. Use `lru_cache` to speed up a recursive function.
3. Create a timezone-aware datetime for a city you care about.

## 9.
L9: `functools.singledispatch` (generic functions)

Define a function with different implementations by type.

In [ ]:

from functools import singledispatch

@singledispatch
def to_text(x) -> str:
    return str(x)

@to_text.register
def _(x: bytes) -> str:
    return x.decode("utf-8", errors="replace")

print(to_text(123))
print(to_text(b"hi"))


## 10.
L10: `statistics`, `decimal`, and `fractions`

Precision + basic stats without third-party packages.

In [ ]:

import statistics
from decimal import Decimal
from fractions import Fraction

print("mean:", statistics.mean([1,2,3,4]))
print("Decimal:", Decimal("0.1") + Decimal("0.2"))
print("Fraction:", Fraction(1, 3) + Fraction(1, 6))
